# Understanding lightweight RGB + IMU fusion

Synthetic examples only. No training, model download or recognition benchmark. Historical results and limitations are documented separately in ../docs/evaluation.md.

In [1]:
import socket
def forbidden(*args, **kwargs):
    raise AssertionError('Network disabled for this notebook')
socket.socket.connect = forbidden
socket.create_connection = forbidden
import numpy as np
import torch
from ai_research_work.models import SensorModel, VideoModel
from ai_research_work.f1 import predict
from ai_research_work.f1.preprocessing import hof
torch.set_num_threads(1)
print('Synthetic offline mode')

Synthetic offline mode


## Shapes and model size
Random initialization demonstrates only the interfaces. Each branch produces eight logits.

In [2]:
s,v=SensorModel(),VideoModel(8,128)
with torch.inference_mode():
    print('Sensor:',tuple(s(torch.zeros(1,128,6)).shape))
    print('HOF:',tuple(v(torch.zeros(1,16,128)).shape))
print('Parameters:',sum(p.numel() for m in (s,v) for p in m.parameters()))

Sensor: (1, 8)
HOF: (1, 8)
Parameters: 66672


## Motion representation
Static synthetic frames yield zero optical-flow histograms. This is a present observation, not a missing modality.

In [3]:
x=hof(np.zeros((2,120,160,3),np.uint8))
print('Histogram:',x.shape,'sum:',float(x.sum()))

Histogram: (16, 128) sum: 0.0


## Equal averaging and surviving branches
Probabilities below are made up for arithmetic illustration, not measured predictions.

In [4]:
ps=np.array([.7,.1,.05,.05,.025,.025,.025,.025])
pv=np.array([.5,.3,.05,.05,.025,.025,.025,.025])
p=.5*ps+.5*pv
print('Both:',p.round(3).tolist())
print('Sensor-only:',ps.tolist())
print('Neither:',predict('absent')['status'])

Both: [0.6, 0.2, 0.05, 0.05, 0.025, 0.025, 0.025, 0.025]
Sensor-only: [0.7, 0.1, 0.05, 0.05, 0.025, 0.025, 0.025, 0.025]
Neither: insufficient_input


## Rejection and descriptive sets
D2 uses 1−max raw p. Unknown rejection precedes C2; a singleton is accepted only after passing the detector. Temperatures change reporting, not these rules. Toy thresholds below are not fitted parameters.

In [5]:
threshold=.5; q=.45
score=1-p.max()
indices=np.flatnonzero(1-p<=q).tolist()
status='unknown' if score>threshold else 'accepted' if len(indices)==1 else 'uncertain'
print({'score':round(float(score),3),'toy_set_indices':indices,'status':status})

{'score': 0.4, 'toy_set_indices': [0], 'status': 'accepted'}


## What this does not establish
C2 is descriptive pooled-crossfit calibration, not a formal new-person guarantee. Throw can be accepted as wave in the historical development evidence. Similar retrieved neighbors do not prove a decision correct. No trained accuracy follows from these cells.